# 01 — Dataset, preprocesamiento resumido y splits congelados

**TP Final · Aprendizaje de Máquina I (CEIA-FIUBA)** · Jaime Pinzón (a2629)

## 1. Propuesta de investigación

**Pregunta:** ¿cuántos días permanecerá internado un paciente diabético, estimado al momento de su ingreso, a partir de la información clínica y administrativa disponible en la admisión?

**Motivación de negocio.** La duración de la estadía (*length of stay*) es uno de los principales determinantes del costo hospitalario y de la disponibilidad de camas. Un modelo con error medio bajo permite planificar ocupación de camas, dimensionar personal por adelantado y detectar tempranamente pacientes con riesgo de estadía prolongada. Por eso la métrica principal de todo el trabajo es el **MAE en días** — directamente interpretable por la gestión — acompañada de RMSE y R².

**Problema de ML:** regresión supervisada sobre el target `time_in_hospital` (1–14 días). En los notebooks siguientes se comparan cinco familias de modelos (KNN, SVR, árbol de regresión, Random Forest, XGBoost) contra baselines simples, con hiperparámetros optimizados por validación cruzada.

## 2. Fuente de datos

> Strack, B., DeShazo, J., Gennings, C., Olmo, J. L., Ventura, S., Cios, K. J., & Clore, J. N. (2014). *Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records*. BioMed Research International. DOI: [10.1155/2014/781670](https://doi.org/10.1155/2014/781670)
>
> Dataset: **Diabetes 130-US Hospitals for Years 1999–2008** — UCI Machine Learning Repository. DOI: [10.24432/C5230J](https://doi.org/10.24432/C5230J) · Licencia: CC BY 4.0

Cada observación es un encuentro hospitalario de un paciente diagnosticado con diabetes, en 130 hospitales de EE.UU. durante 1999–2008: 101.766 registros y 50 variables (datos demográficos, diagnósticos ICD-9, medicaciones, utilización previa del sistema de salud).

El análisis exploratorio completo y las decisiones de preprocesamiento provienen del trabajo final de **Análisis de Datos** (Grupo 3, mismo autor): [CEIA_Analisis_de_datos/trabajo_final](https://github.com/masouto94/CEIA_Analisis_de_datos). Siguiendo el criterio de la cátedra, aquí solo se **resume** ese trabajo y el foco está en el entrenamiento y la evaluación.

In [1]:
import pandas as pd

from src.config import (
    COLS_FILTRO_IQR, DIR_PROCESSED, FEATURES_FINALES, SEED, TARGET,
)
from src.preprocesamiento import (
    cargar_y_limpiar, descargar_si_falta, filtrar_outliers_train,
    procesar, split_estratificado,
)
from src.pipelines import build_preprocessor

pd.set_option("display.max_columns", 60)
SEED

42

## 3. Preprocesamiento resumido (portado de Análisis de Datos)

Tabla-síntesis de las decisiones tomadas en AdD y qué se porta acá. La regla del porte: los parquets finales contienen **solo transformaciones sin estado**; cualquier transformación que aprende de los datos (OneHot, TargetEncoder, escalado) vive dentro del `Pipeline` de cada modelo (`src/pipelines.py`) y se re-fitea por fold en validación cruzada.

| Decisión en AdD | Justificación original | ¿Se porta? |
|---|---|---|
| `weight` eliminada | 97% faltante (tema legislativo en EE.UU.) | No hace falta: no sobrevive la selección |
| `max_glu_serum`, `A1Cresult` → "desconocido" | Ausentes estructurales (test no realizado) | Ídem |
| `race` → "Other", `payer_code`/`medical_specialty` → "Unknown" | MCAR / faltantes masivos | Ídem |
| Cascada de diagnósticos `diag_1 > diag_2 > diag_3` | Faltantes ≤1%, imputación por jerarquía clínica | **Sí** |
| Mapeo ICD-9 → 21 categorías clínicas | Reducción de cardinalidad (~700 códigos → 21) | **Sí** (determinista, sin estado) |
| `examide`, `citoglipton` descartadas | Varianza cero | No hace falta: no sobreviven la selección |
| Split 80/20, seed 42, estratificado por target | Target discreto (1–14), preserva distribución | **Sí** (idéntico) |
| Filtro IQR de outliers **solo en train** | Reduce asimetría; test intocado | **Sí** (idéntico) |
| `in_out_emergency` (feature nueva) | Condensa visitas previas | **No**: la selección final de AdD no la retuvo (código muerto) |
| Selección por información mutua + correlaciones | 8 variables base retenidas | **Sí** (lista fija `FEATURES_FINALES`) |
| OneHot(`age`), TargetEncoder(`diag_*`) | Codificación por cardinalidad | **Sí, pero dentro del Pipeline** (nunca en los parquets) |
| PCA (11 de 17 componentes) | Reducción de dimensionalidad | **No** en el pipeline principal (decisión D1 del plan); posible anexo |

**Nota de fidelidad:** en el original, la línea que reemplazaba `?` en `diag_1` no asignaba el resultado (no-op). Es inocuo: el mapeo ICD-9 envía tanto `?` como `None` a la categoría 0. El porte replica el comportamiento efectivo.

Las 8 variables finales: `num_lab_procedures`, `num_procedures`, `num_medications`, `number_diagnoses` (numéricas), `age` (categórica ordinal en rangos de 10 años) y `diag_1/2/3` (categorías ICD-9).

In [2]:
# Descarga (si hace falta) y carga del crudo
descargar_si_falta()
from src.config import CSV_CRUDO
crudo = pd.read_csv(CSV_CRUDO)
print(f"Dataset crudo: {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
crudo[FEATURES_FINALES + [TARGET]].head(3)

Dataset crudo: 101,766 filas x 50 columnas


,num_lab_procedures,num_procedures,num_medications,number_diagnoses,age,diag_1,diag_2,diag_3,time_in_hospital
0,41,0,1,1,[0-10),250.83,?,?,1
1,59,0,18,9,[10-20),276,250.01,255,3
2,11,5,13,6,[20-30),648,250,V27,2


In [3]:
# Limpieza sin estado: cascada de diagnosticos + mapeo ICD-9 + dtypes
X, y = cargar_y_limpiar()
print(f"X: {X.shape} (8 features finales + 3 columnas auxiliares del filtro IQR)")
print(f"y: {y.shape}, rango {y.min():.0f}-{y.max():.0f} dias")
X.head(3)

X: (101766, 11) (8 features finales + 3 columnas auxiliares del filtro IQR)
y: (101766,), rango 1-14 dias


,num_lab_procedures,num_procedures,num_medications,number_diagnoses,age,diag_1,diag_2,diag_3,number_outpatient,number_emergency,number_inpatient
0,41,0,1,1,[0-10),20,20,20,0,0,0
1,59,0,18,9,[10-20),3,20,3,0,0,0
2,11,5,13,6,[20-30),11,20,18,2,0,1


## 4. Split estratificado 80/20

Idéntico al de AdD: `random_state=42`, estratificado por el target. Como `time_in_hospital` es discreto (1–14 días), estratificar garantiza que train y test tengan la misma distribución de estadías — importante porque la distribución es asimétrica (mayoría de estadías cortas).

In [4]:
X_train_full, X_test_full, y_train_full, y_test = split_estratificado(X, y)

comparacion = pd.DataFrame({
    "original": y.value_counts(normalize=True).sort_index(),
    "train": y_train_full.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).round(4)
print(f"train: {X_train_full.shape[0]:,} filas | test: {X_test_full.shape[0]:,} filas")
comparacion.T

train: 81,412 filas | test: 20,354 filas


time_in_hospital,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0
original,0.1396,0.1693,0.1745,0.1368,0.0979,0.0741,0.0576,0.0431,0.0295,0.023,0.0182,0.0142,0.0119,0.0102
train,0.1396,0.1693,0.1745,0.1368,0.0979,0.0741,0.0576,0.0432,0.0295,0.023,0.0182,0.0142,0.0119,0.0102
test,0.1396,0.1693,0.1745,0.1368,0.0979,0.0741,0.0576,0.0431,0.0295,0.023,0.0182,0.0142,0.0119,0.0102


## 5. Filtro de outliers IQR — solo sobre train

Los umbrales $[Q_1 - 1.5\,\mathrm{IQR},\; Q_3 + 1.5\,\mathrm{IQR}]$ se calculan **sobre el train** para las 7 columnas numéricas y el target, y se conservan las filas donde todas las columnas están dentro de sus límites (réplica exacta de AdD, que marcaba outliers con NaN y hacía un único `dropna`).

**El test no se filtra jamás**: en producción no se puede descartar un paciente por tener valores extremos, así que la evaluación debe incluirlos. Esto implica que el test conserva estadías de hasta 14 días aunque el train quede acotado — los modelos serán evaluados también sobre casos que no vieron en entrenamiento, y eso es deliberado y honesto.

In [5]:
X_train, y_train = filtrar_outliers_train(X_train_full, y_train_full)
X_test = X_test_full[FEATURES_FINALES].reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"train filtrado: {X_train.shape} | y_train: {y_train.shape}")
print(f"test SIN filtrar: {X_test.shape} | y_test: {y_test.shape}")
print(f"y_train: {y_train.min():.0f}-{y_train.max():.0f} dias | y_test: {y_test.min():.0f}-{y_test.max():.0f} dias")

# Verificacion contra los shapes del trabajo original de AdD
assert X_train.shape == (53756, 8), X_train.shape
assert y_train.shape == (53756,), y_train.shape
assert X_test.shape == (20354, 8), X_test.shape
assert y_test.shape == (20354,), y_test.shape
print("Shapes identicos a los del trabajo de AdD (53.756 train / 20.354 test)")

train filtrado: (53756, 8) | y_train: (53756,)
test SIN filtrar: (20354, 8) | y_test: (20354,)
y_train: 1-12 dias | y_test: 1-14 dias
Shapes identicos a los del trabajo de AdD (53.756 train / 20.354 test)


## 6. Reproducibilidad y persistencia

Criterio del plan (tarea 1.6): dos ejecuciones independientes del pipeline completo deben producir DataFrames **idénticos** (`pd.testing.assert_frame_equal`), y los parquets deben contener datos limpios **sin codificar**.

In [6]:
# Dos ejecuciones independientes de todo el pipeline -> resultados identicos
datos_a = procesar()
datos_b = procesar()

pd.testing.assert_frame_equal(datos_a["X_train"], datos_b["X_train"])
pd.testing.assert_frame_equal(datos_a["X_test"], datos_b["X_test"])
pd.testing.assert_series_equal(datos_a["y_train"], datos_b["y_train"])
pd.testing.assert_series_equal(datos_a["y_test"], datos_b["y_test"])

# ... y tambien identicos a lo construido paso a paso mas arriba
pd.testing.assert_frame_equal(datos_a["X_train"], X_train)
pd.testing.assert_series_equal(datos_a["y_test"], y_test)
print("Reproducibilidad verificada: dos corridas independientes -> DataFrames identicos")

Reproducibilidad verificada: dos corridas independientes -> DataFrames identicos


In [7]:
# Persistencia en parquet (datos limpios SIN codificar)
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)
X_train.to_parquet(DIR_PROCESSED / "X_train.parquet")
X_test.to_parquet(DIR_PROCESSED / "X_test.parquet")
y_train.to_frame().to_parquet(DIR_PROCESSED / "y_train.parquet")
y_test.to_frame().to_parquet(DIR_PROCESSED / "y_test.parquet")

# Verificacion 1: roundtrip sin perdida
rt = pd.read_parquet(DIR_PROCESSED / "X_train.parquet")
pd.testing.assert_frame_equal(rt, X_train)

# Verificacion 2: SIN columnas codificadas ni escaladas
assert list(rt.columns) == FEATURES_FINALES, "columnas inesperadas en el parquet"
assert not any(c.startswith("age_") for c in rt.columns), "hay columnas OneHot en el parquet"
assert rt["age"].dtype == "category", "age debe seguir siendo categorica cruda"
assert rt[["diag_1", "diag_2", "diag_3"]].isin(range(0, 21)).all().all(), \
    "diag_* deben ser categorias ICD-9 enteras (0-20), no valores target-encoded"
print("Parquets escritos y verificados: 8 columnas limpias, sin codificacion, sin escalado")

Parquets escritos y verificados: 8 columnas limpias, sin codificacion, sin escalado


## 7. Checklist anti-fuga de datos

| Control | Estado |
|---|---|
| El filtro IQR calcula cuantiles SOLO sobre train (`filtrar_outliers_train` recibe únicamente `X_train, y_train`) | ✅ |
| El test nunca se filtra (solo selección de columnas y `reset_index`) | ✅ |
| Los parquets contienen datos limpios sin codificar (verificado arriba con asserts) | ✅ |
| Ningún objeto con `fit` vio `X_test`/`y_test` en este notebook: el único `fit` (celda siguiente) es del preprocesador de demostración, fiteado con train; sobre test solo se llama `transform` | ✅ |
| OneHot/TargetEncoder/StandardScaler viven en `build_preprocessor()` dentro del Pipeline de cada modelo → en CV se re-fitean por fold | ✅ |
| El `TargetEncoder` usa CV interno (KFold 5, seed 42): ni siquiera el train se codifica directamente con su propio target | ✅ |

In [8]:
# Vista previa del preprocesador comun (el que usaran TODOS los modelos).
# fit SOLO con train; sobre test unicamente transform.
prep = build_preprocessor(scale=False)
Xt_train = prep.fit_transform(X_train, y_train)
Xt_test = prep.transform(X_test)

nombres = list(prep.get_feature_names_out())
print(f"train: {X_train.shape} -> {Xt_train.shape} | test: {X_test.shape} -> {Xt_test.shape}")
print(f"{len(nombres)} columnas efectivas:")
print(nombres)

# Mismas 17 columnas efectivas que la matriz final del trabajo de AdD
esperadas = {
    "diag_1", "diag_2", "diag_3",
    "num_medications", "num_lab_procedures", "number_diagnoses", "num_procedures",
    "age_[0-10)", "age_[10-20)", "age_[20-30)", "age_[30-40)", "age_[40-50)",
    "age_[50-60)", "age_[60-70)", "age_[70-80)", "age_[80-90)", "age_[90-100)",
}
assert set(nombres) == esperadas and len(nombres) == 17
print("Coincide con las 17 columnas del trabajo de AdD")

train: (53756, 8) -> (53756, 17) | test: (20354, 8) -> (20354, 17)
17 columnas efectivas:
['age_[0-10)', 'age_[10-20)', 'age_[20-30)', 'age_[30-40)', 'age_[40-50)', 'age_[50-60)', 'age_[60-70)', 'age_[70-80)', 'age_[80-90)', 'age_[90-100)', 'diag_1', 'diag_2', 'diag_3', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']
Coincide con las 17 columnas del trabajo de AdD


## 8. Cierre

Quedaron congelados los cuatro parquets en `data/processed/` — el único origen de datos para los notebooks 02–07:

- `X_train` 53.756 × 8 (filtrado por IQR de train), `y_train` (1–12 días tras el filtro)
- `X_test` 20.354 × 8 (**sin filtrar**), `y_test` (1–14 días)

**Siguiente notebook (02):** baselines — el piso que cualquier modelo debe superar.